# Automated ML Project Architect

In [31]:
# Importing Necessary Libraries:

import os
from dotenv import load_dotenv
import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()

## Step-1: Pandas Extractor

In [2]:
data = pd.read_csv('loan_borowwer_data.csv')
data.head()

,credit.policy,purpose,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0


In [14]:
# Function to Load in CSV and Getting Columns, Data-Types and First Few Rows:

def data_profile(file_path):

    # Loading Given CSV File:
    df = pd.read_csv(file_path)

    # Getting Column Names:
    column_names = df.columns.to_list()
    #print(column_names)

    # Getting Data Types:
    data_types = df.dtypes.to_list()
    #print(data_types)

    # Getting First few Rows for Example Data:
    data_sample = df.head().to_string(header= False)
    #print(data_sample)

    # Getting Missing value Counts:
    missing_count = df.isnull().sum().to_string()
    #print(missing_count)

    # Getting Summary Statistics:
    summary_stats = df.describe().to_string()
    #print(summary_stats)

    return column_names, data_types, data_sample, missing_count, summary_stats

In [15]:
# Testing Above function:
a,b,c,d,e = data_profile(file_path= 'C://Users//shail//PycharmProjects//PythonProject//Applied-LLM-Engineering//01_Exploring_Top_Models//loan_borowwer_data.csv')

In [16]:
print(a, b, c, d, e)

['credit.policy', 'purpose', 'int.rate', 'installment', 'log.annual.inc', 'dti', 'fico', 'days.with.cr.line', 'revol.bal', 'revol.util', 'inq.last.6mths', 'delinq.2yrs', 'pub.rec', 'not.fully.paid'] [dtype('int64'), dtype('O'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('int64'), dtype('float64'), dtype('int64'), dtype('float64'), dtype('int64'), dtype('int64'), dtype('int64'), dtype('int64')] 0  1  debt_consolidation  0.1189  829.10  11.350407  19.48  737  5639.958333  28854  52.1  0  0  0  0
1  1         credit_card  0.1071  228.22  11.082143  14.29  707  2760.000000  33623  76.7  0  0  0  0
2  1  debt_consolidation  0.1357  366.86  10.373491  11.63  682  4710.000000   3511  25.6  1  0  0  0
3  1  debt_consolidation  0.1008  162.34  11.350407   8.10  712  2699.958333  33667  73.2  1  0  0  0
4  1         credit_card  0.1426  102.92  11.299732  14.97  667  4066.000000   4740  39.5  0  1  0  0 credit.policy        0
purpose              0
int.rate    

In [32]:
# Writing a Function for API Call to LLM:

def pandas_extractor(column_names, data_types, data_sample, missing_count, summary_stats):

    # Defining System prompt:
    system_prompt = """
    You are a Senior Data Scientist. Your Job is to analyse the given Data Profile and help the user with deciding Data Cleaning Process and Problem Type and Model Recommendations.

    Output a JSON object containing the target_variable, problem_type, recommended_models, and specific data_cleaning_warnings.

    Output Should be in this Specific JSON Format:
    {
    "target_variable": "Churn",
    "problem_type": "Binary Classification",
    "recommended_models": ["Logistic Regression", "Support Vector Machine"],
    "data_cleaning_warnings": ["'Date' column is a string", "'Age' has missing values"]
    }
    """

    # Defining User prompt:
    user_prompt = f"""
    Here are the column names:
    {column_names}

    Here are the data types:
    {data_types}

    Here is a sample of the data:
    {data_sample}

    Here is the count of missing values per column:
    {missing_count}

    Here are the summary statistics:
    {summary_stats}

    Based on this complete data profile, please help me decide data cleaning steps, problem type and model recommendations.

    """

    # Defining Message:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]

    # Calling LLM API:
    response = openai.chat.completions.create(model= 'gpt-5-mini',
                                              messages= messages,
                                              response_format= {'type': 'json_object'})

    return response.choices[0].message.content


In [33]:
# Testing pandas_extractor Function:
column_names, data_types, data_sample, missing_count, summary_stats = data_profile('loan_borowwer_data.csv')

test_res = pandas_extractor(column_names= column_names,
                            data_types= data_types,
                            data_sample= data_sample,
                            missing_count= missing_count,
                            summary_stats= summary_stats)

In [34]:
print(test_res)

{
  "target_variable": "not.fully.paid",
  "problem_type": "Binary Classification",
  "recommended_models": [
    "Logistic Regression (with class_weight or regularization)",
    "Random Forest",
    "Gradient Boosting (XGBoost or LightGBM)",
    "CatBoost (handles categorical 'purpose' well)",
    "Support Vector Machine (with scaling) — for smaller experiments"
  ],
  "data_cleaning_warnings": [
    "'purpose' is object/categorical — requires encoding (one-hot, target/frequency encoding, or use CatBoost's native handling)",
    "'revol.bal' is highly skewed with a very large max (~1.2e6) — check for outliers, consider capping or log-transform",
    "'installment' and 'revol.bal' distributions are skewed — consider transformation or robust scaling for some models",
    "'log.annual.inc' is already log-transformed — do not re-log; consider exponentiating only for interpretability if needed",
    "'int.rate' is expressed as a decimal (e.g. 0.1189 = 11.89%) — ensure consistent interpreta

## Step-2: The Tech Writer